In [ ]:
# Import packages
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from IPython.display import display, HTML

In [ ]:
# Define the Raw GitHub Data Repository URLs
batter_url = "https://github.com/willcreager/willcreager.github.io/raw/refs/heads/main/docs/data/csvs/MLBData-ABSSituationalSplits-Batters-FullSeason.csv"
catcher_url = "https://github.com/willcreager/willcreager.github.io/raw/refs/heads/main/docs/data/csvs/MLBData-ABSSituationalSplits-Catchers-FullSeason.csv"

In [ ]:
def run_pca_pipeline(url_path, position_label, min_opportunities=100):
    print(f"\n==================================================================")
    print(f"📉 RUNNING PCA & SVD DIMENSIONALITY PIPELINE FOR: {position_label.upper()}")
    print(f"==================================================================")

    # Load Remote Dataset
    try:
        df = pd.read_csv(url_path, encoding="latin1")
    except Exception as e:
        print(f"Error fetching data: {e}")
        return

    # ---- DATA CLEANING & THRESHOLD FILTERING ----
    # Filter out players with tiny sample sizes to remove statistical noise from rates
    df_filtered = df[df["Total_Opp_Ovr"] >= min_opportunities].copy().reset_index(drop=True)
    print(f"Original dataset rows: {len(df)} | Rows after filtering (Total_Opp_Ovr >= {min_opportunities}): {len(df_filtered)}")

    if len(df_filtered) < 5:
        print("❌ Not enough data rows remaining after filtering. Lower your min_opportunities constraint.")
        return

    # ISOLATION RULE: Keep columns ending in _Rate, but strictly block anything containing 'Ovr' or 'Total'
    rate_features = [col for col in df_filtered.columns if col.endswith("_Rate")]
    rate_features = [col for col in rate_features if "Ovr" not in col and "Total" not in col]

    print(f"Isolating all {len(rate_features)} aggression features for clustering calculation:")

    # Fill any remaining NaNs with 0.0 (e.g., if a player had 0 extra-inning opportunities)
    X = df_filtered[rate_features].fillna(0.0)

    # Standardize Features (Crucial for PCA to ensure equal variance weighting)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Get screenshot of final data used for model
    scaled_df = pd.DataFrame(data=X_scaled, columns=rate_features)
    print(scaled_df.head())

    # ---- COMPUTE PCA & EXTRACT EIGENVALUES ----
    pca = PCA(random_state=42)
    X_pca = pca.fit_transform(X_scaled)

    # Calculate Explained Variance Percentages
    percent_variance = np.round(pca.explained_variance_ratio_ * 100, decimals=2)
    cum_variance = np.cumsum(pca.explained_variance_ratio_ * 100)

    # ---- VISUALIZATION 1: SCREE PLOT ----
    plt.figure(figsize=(10, 5))
    max_display = min(12, len(percent_variance))
    plt.bar(range(1, max_display + 1), percent_variance[:max_display], alpha=0.6, align="center", label="Individual Variance Captured", color="navy")
    plt.step(range(1, max_display + 1), cum_variance[:max_display], where="mid", label="Cumulative Variance Captured", color="crimson", linewidth=2)

    # Draw reference line at 80% variance target
    plt.axhline(y=80, color="gray", linestyle="--", alpha=0.7, label="80% Variance Threshold")

    plt.ylabel("Percentage of Explained Variance (%)", fontsize=11, fontweight="bold")
    plt.xlabel("Principal Components", fontsize=11, fontweight="bold")
    plt.title(f"Scree Plot: {position_label}", fontsize=13, fontweight="bold")
    plt.xticks(range(1, max_display + 1))
    plt.legend(loc="lower right")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"ℹ️ PC1 explains {percent_variance[0]:.2f}% of the variance.")
    print(f"ℹ️ PC2 explains {percent_variance[1]:.2f}% of the variance.")
    print(f"ℹ️ Cumulative variance captured by the first two components: {cum_variance[1]:.2f}%")

    # ---- VISUALIZATION 2: 2D PRINCIPAL COMPONENT SPACE ----
    pca_df = pd.DataFrame(data=X_pca[:, :2], columns=["PC1", "PC2"])
    pca_df["PlayerName"] = df_filtered["firstName"] + " " + df_filtered["lastName"]
    pca_df["Total_Challenges"] = df_filtered["Total_ChallengesMade"]
    pca_df["Total_OverturnedRate"] = df_filtered["Total_OverturnedRate"]

    plt.figure(figsize=(9, 6))
    # Color-code points by their overturn rate to check if PCA clusters performance metrics
    scatter = plt.scatter(pca_df["PC1"], pca_df["PC2"], c=pca_df["Total_OverturnedRate"], cmap="coolwarm", s=70, alpha=0.75, edgecolors="black", linewidth=0.5)
    cbar = plt.colorbar(scatter)
    cbar.set_label("Overturn Rate (%)", fontsize=10, fontweight="bold")

    plt.xlabel(f"Principal Component 1", fontsize=11, fontweight="bold")
    plt.ylabel(f"Principal Component 2", fontsize=11, fontweight="bold")
    plt.title(f"2D Projection onto PC Space: {position_label}", fontsize=13, fontweight="bold")
    plt.axhline(0, color="black", linestyle="--", alpha=0.3)
    plt.axvline(0, color="black", linestyle="--", alpha=0.3)
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

    # ---- FEATURE LOADINGS WEIGHT MATRIX ----
    # Extract the alignment weights that link features back to components
    loadings = pd.DataFrame(pca.components_.T, columns=[f"PC{i+1}" for i in range(len(rate_features))], index=rate_features)

    print(f"\n📋 TOP FEATURE LOADINGS FOR {position_label.upper()}:")

    # Build a clean data framework displaying weights side-by-side
    loadings_summary = pd.DataFrame({
        "PC1_Weight": loadings["PC1"],
        "PC2_Weight": loadings["PC2"]
    }).round(3)

    # Generate stylized HTML table for the user's web page deployment code
    html_markup = loadings_summary.sort_values(by="PC1_Weight", key=abs, ascending=False).to_html(classes="table table-striped table-hover", border=1)

    print("\n--- COPY AND PASTE THIS FEATURE LOADINGS HTML MATRIX INTO YOUR PCA.HTML PAGE ---")
    print(html_markup)
    print("--------------------------------------------------------------------------------\n")

    display(HTML(f"<h3>Eigenvector Feature Weights: {position_label}</h3>"))
    display(HTML(html_markup))

In [ ]:
# Execute the analysis pipeline across your repository data links
# Set min_opportunities=15 to screen out bench players with only 1 or 2 challenge instances
run_pca_pipeline(url_path=batter_url, position_label="Batters", min_opportunities=100)

In [ ]:
# Execute the analysis pipeline across your repository data links
# Set min_opportunities=15 to screen out bench players with only 1 or 2 challenge instances
run_pca_pipeline(url_path=catcher_url, position_label="Catchers", min_opportunities=100)

# Create diagrams for website

In [ ]:
# ==============================================================================
# DIAGRAM 1: EIGENVECTOR ROTATION & VARIANCE CAPTURE
# ==============================================================================
np.random.seed(42)
# Generate a synthetic correlated cluster of data points
x = np.random.normal(0, 1, 150)
y = 1.5 * x + np.random.normal(0, 0.4, 150)
data = np.vstack([x, y]).T

fig, ax1 = plt.subplots(figsize=(7, 6))

# Plot the underlying data cloud
ax1.scatter(data[:, 0], data[:, 1], color="#495057", alpha=0.5, edgecolors="none", s=25)

# Define the primary and secondary structural principal axes
origin = [0, 0]
v1 = np.array([1, 1.5]) / np.sqrt(1 + 1.5**2) * 2.5
v2 = np.array([-1.5, 1]) / np.sqrt(1.5**2 + 1) * 1.0

# Draw explicit eigenvector paths using quiver arrows
ax1.quiver(*origin, *v1, scale=1, scale_units="xy", color="crimson", width=0.012, zorder=5, label="PC1 (Direction of Maximum Spread)")
ax1.quiver(*origin, *v2, scale=1, scale_units="xy", color="royalblue", width=0.012, zorder=5, label="PC2 (Orthogonal Axis)")

# Annotate the mean center anchor point
ax1.scatter(0, 0, color="yellow", edgecolors="black", s=100, marker="o", zorder=6, label="Data Center (Mean)")

# Labeling and layout adjustments
ax1.set_xlim(-4, 4)
ax1.set_ylim(-4, 4)
ax1.axhline(0, color="black", linewidth=0.8, alpha=0.3)
ax1.axvline(0, color="black", linewidth=0.8, alpha=0.3)
ax1.set_xlabel("Standardized Feature X", fontweight="semibold")
ax1.set_ylabel("Standardized Feature Y", fontweight="semibold")
ax1.set_title("Eigenvector Projections & Rotations Space", fontsize=12, fontweight="bold", pad=12)
ax1.legend(loc="upper left", frameon=True, fontsize=9)
ax1.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# DIAGRAM 2: DIMENSIONAL SPACE REDUCTION CONCEPT
# ==============================================================================
np.random.seed(42)
fig, ax2 = plt.subplots(figsize=(7, 6))

# Generate another subset cloud to show orthogonal projection lines
xp = np.random.uniform(-3, 3, 20)
yp = 0.8 * xp + np.random.normal(0, 0.5, 20)

# Plot the rotated PC1 axis baseline line equation
line_x = np.linspace(-4, 4, 100)
line_y = 0.8 * line_x
ax2.plot(line_x, line_y, color="crimson", linewidth=2, label="1D Principal Axis Component")

# Map out individual row items alongside their straight-line projections
for i in range(len(xp)):
    # Project each coordinate down perpendicular onto the line path
    denom = 1.0 + 0.8**2
    proj_x = (xp[i] + 0.8 * yp[i]) / denom
    proj_y = 0.8 * proj_x

    # Draw gray dashed connectors showing the dropped informational details
    ax2.plot([xp[i], proj_x], [yp[i], proj_y], color="dimgray", linestyle="--", alpha=0.5, linewidth=1.2)
    ax2.scatter(proj_x, proj_y, color="crimson", s=30, zorder=4)

# Draw original continuous matrix coordinate points
ax2.scatter(xp, yp, color="#495057", edgecolors="black", s=50, zorder=5, label="Original Data Point")

# Labeling and layout adjustments
ax2.set_xlim(-4, 4)
ax2.set_ylim(-4, 4)
ax2.axhline(0, color="black", linewidth=0.8, alpha=0.3)
ax2.axvline(0, color="black", linewidth=0.8, alpha=0.3)
ax2.set_xlabel("Feature X", fontweight="semibold")
ax2.set_ylabel("Feature Y", fontweight="semibold")
ax2.set_title("Dimensional Space Reduction: Orthogonal Linear Projection", fontsize=12, fontweight="bold", pad=12)
ax2.legend(loc="upper left", frameon=True, fontsize=9)
ax2.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()